**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/tingley-2019"], check=True)
    os.chdir("/content/encore/entries/tingley-2019")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Tingley and Buzsaki (2019) from DANDI:000218: Figure 7E

Tingley D. and Buzsaki G. (2019). Routing of hippocampal ripples to subcortical structures via the lateral septum. Neuron 105, 138-149. doi:10.1016/j.neuron.2019.10.012.

Dataset: DANDI:000218, version 0.261110.1911 (doi:10.48324/dandi.000218/0.261110.1911), streamed from the archive.

Version. The paper cites no version of the dandiset (it predates it). The most recent published version, 0.261110.1911, is pinned: its files were re-converted with trial labels sorted with their rows, the lab's ripple and LS HFO detections, sleep states for every session with them, units with shank and region, and electrode flags, all of which the claims use; the earlier versions (0.220131.1608, 0.250624.0439) lack the event tables and sleep states, so they match the claims file's description less well. The LFP is the earlier version's data, stored again in channel order (release notes).

Claims file: `claims/tingley-2019.yaml`, written from the accepted manuscript in PubMed Central (PMC6952543) with its supplement. The paper's full text and supplement were read here.

This notebook tests the Figure 7E claims that the dataset allows: LS spike counts in SPW-Rs predicted from the hippocampal spike count, the brain state and the SPW-R duration.

Claims and where they are tested (all notebooks of this entry):

| Claim | Where tested |
|---|---|
| dataset-composition | main notebook (counts); LFP notebook (LS neurons with more than 50 spikes) |
| fig1c-hfo-each-animal, fig1c-post-burst-suppression, fig1c-event-count, figs1-participation | main notebook |
| fig1d-phase-locking | LFP notebook; own-shank notebook (shanks outside the detection channel's storage block) |
| fig1f-shorter-than-ripples, fig1g-slower-than-ripples, fig1h-no-amplitude-frequency-correlation | main notebook |
| fig2-confined-to-ls | LFP notebook |
| fig2ef-intermediate-ls | blocked: metadata_insufficient (no relative depth of each recording within LS) |
| fig3-travelling-waves | blocked: metadata_insufficient (no electrode positions) |
| fig4b-pre-post-unchanged, fig4b-hfo-above-behav, fig4-sessions-few-change, fig4c-temporal-bias, fig4d-compression | main notebook |
| figs3d-hpc-compression-post, figs2-post-rate-increase, figs2c-no-ls-reactivation | main notebook |
| fig5b-nrem-and-rates, fig5e-state-coupling, fig5f-ripple-triggered-firing | main notebook |
| fig5d-lag | main notebook (deposited peak times); LFP notebook (maximum-NSS times) |
| fig5f-no-phase-coherence, ls-spikes-not-locked-to-ca1-ripples | LFP notebook |
| fig6-optogenetic-hfos | blocked: data_missing (the two optogenetic rats are not in the dandiset) |
| fig7e-ca1-spikes, fig7e-state, fig7e-duration | Figure 7E notebook |
| fig7e-ca1-depth | blocked: metadata_insufficient (no site positions or per-site waveforms) |
| fig7e-replay, fig7e-seqnmf, figs5-seqnmf-variance | blocked: other (not attempted within the run's time) |
| fig7e-content-independent, fig7e-coupled-only | blocked: other (the content features, fig7e-replay and fig7e-seqnmf, were not attempted) |
| discussion-medial-septum | main notebook (part b); LFP notebook (part a) |
| discussion-cingulate-down-state | main notebook (reading a) |
| fig8-model | blocked: other (the model's data are elsewhere and it was not attempted within the run's time) |

## Setup
Every file of the pinned version is streamed; from each, the spikes, units, event tables, sleep states,
trials and electrode flags are read into memory. The LFP is not read in this notebook.

In [ ]:
import zlib, time, resource, collections, warnings
import numpy as np, pandas as pd
from scipy import stats
import matplotlib.pyplot as plt
import tingley2019 as T
warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 200)
KEY = "tingley-2019"
def seed_of(claim): return zlib.crc32(f"{KEY}/{claim}".encode())
t0 = time.time()
A = T.list_assets()
S = T.load_all(A, workers=8)
print(f"{len(S)} files of dandiset {T.DANDISET} version {T.VERSION} loaded in {time.time()-t0:.0f} s; "
      f"peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss/1e6:.2f} GB")

## Data check
What was loaded, by subject and session; the ranges and units of every variable used; recordings deposited
twice; and whether the spikes, the event tables, the sleep states and the trials share one clock.

In [ ]:
def ls_ev(s, fallback=False):
    """The lab's main LS HFO detection (LSRipples); with fallback, the earlier ls_ripples, then ripples_ls."""
    for k in (["LSRipples", "ls_ripples", "ripples_ls"] if fallback else ["LSRipples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
def ca1_ev(s, fallback=False):
    """The lab's CA1 ripples (CA1Ripples); with fallback, the hippocampal ripplesALL."""
    for k in (["CA1Ripples", "ripplesALL"] if fallback else ["CA1Ripples"]):
        if k in s.events and len(s.events[k]):
            return s.events[k]
    return None
HPC = list(T.HPC_REGIONS)
rows = []
for s in S:
    reg = collections.Counter(s.unit_region)
    sl = {} if s.sleep is None else (s.sleep.stop_time - s.sleep.start_time).groupby(s.sleep.label).sum().to_dict()
    rows.append(dict(subject=s.subject, session=s.session[:30], hours=round(s.duration/3600, 2), lfp_channels=s.lfp_shape[1],
                     ls_units=reg.get("ls", 0), hpc_units=reg.get("hpc", 0), ca1_units=reg.get("ca1", 0), ca3_units=reg.get("ca3", 0),
                     other_units=sum(v for k, v in reg.items() if k not in ("ls", "hpc", "ca1", "ca3")),
                     other_regions=",".join(sorted(k if k else "''" for k in reg if k not in ("ls", "hpc", "ca1", "ca3"))),
                     tables=",".join(f"{k}:{len(v)}" for k, v in sorted(s.events.items())),
                     nrem_h=round(sl.get("NREM", 0)/3600, 2), wake_h=round(sl.get("WAKE", 0)/3600, 2), rem_h=round(sl.get("REM", 0)/3600, 2),
                     trials=0 if s.trials is None else len(s.trials),
                     ca3_flag=bool(s.electrodes.ca3_channel.any()), ca1_flag=bool(s.electrodes.ca1_channel.any())))
INV = pd.DataFrame(rows)
print(INV.to_string())
print("\nBy subject:")
print(INV.groupby("subject")[["ls_units", "hpc_units", "ca3_units", "other_units", "trials"]].sum()
      .join(INV.groupby("subject").size().rename("files")).to_string())
print("\nUnit regions over all files:", dict(collections.Counter(r for s in S for r in s.unit_region)))
print("Event tables (files holding each):", dict(collections.Counter(k for s in S for k in s.events)))

In [ ]:
# Ranges and units of the variables used.
for k in ["LSRipples", "CA1Ripples", "ripplesALL", "ls_ripples", "ripples_ls"]:
    d = pd.concat([s.events[k] for s in S if k in s.events])
    dur = (d.stop_time - d.start_time) * 1000
    inside = np.mean([((s.events[k].start_time >= 0) & (s.events[k].stop_time <= s.duration + 1)).mean() for s in S if k in s.events])
    pk = ((d.peak_time >= d.start_time) & (d.peak_time <= d.stop_time)).mean()
    fr = f"peak_frequency {d.peak_frequency.min():.1f} to {d.peak_frequency.max():.1f} (median {d.peak_frequency.median():.1f}, read as Hz)" if "peak_frequency" in d else "no peak_frequency"
    print(f"{k}: {len(d)} events; duration {dur.min():.1f} to {dur.max():.1f} ms (median {dur.median():.1f}); {fr}; "
          f"peak inside start-stop {pk:.3f}; events inside the recording {inside:.3f}")
allsp = [(s.spikes[i].min(), s.spikes[i].max(), s.duration) for s in S for i in range(len(s.spikes)) if len(s.spikes[i])]
print(f"Spike times: {sum(len(x) for s in S for x in s.spikes)} spikes in {sum(len(s.spikes) for s in S)} units; "
      f"min {min(a for a,_,_ in allsp):.4f} s; units with a spike after the LFP's end: {sum(b > d + 1 for _, b, d in allsp)}")
print("Sorted spike times in every unit:", all(np.all(np.diff(x) >= 0) for s in S for x in s.spikes))
sl = pd.concat([s.sleep.assign(dur=s.duration) for s in S if s.sleep is not None])
print(f"Sleep states: labels {sorted(sl.label.unique())}; bouts {len(sl)}; stop beyond the recording: {(sl.stop_time > sl.dur + 1).sum()}; "
      f"whole-second times: {np.mean(np.isclose(sl.start_time % 1, 0)):.3f}")
tr = pd.concat([s.trials.assign(dur=s.duration) for s in S if s.has_trials])
print(f"Trials: {len(tr)} in {sum(s.has_trials for s in S)} files; length {(tr.stop_time-tr.start_time).min():.2f} to "
      f"{(tr.stop_time-tr.start_time).max():.1f} s (median {(tr.stop_time-tr.start_time).median():.1f}); beyond the recording: {(tr.stop_time > tr.dur + 1).sum()}")

In [ ]:
# Recordings deposited twice: identical spike trains (unit spike counts and first spikes) or identical event tables.
fp_sp, fp_ev, dup = {}, {}, []
for i, s in enumerate(S):
    if len(s.spikes):
        key = tuple(sorted((len(x), round(float(x[0]), 4) if len(x) else 0) for x in s.spikes))
        if key in fp_sp: dup.append(("spikes", S[fp_sp[key]].session, s.session))
        fp_sp[key] = i
    for k, e in s.events.items():
        key = (len(e), tuple(np.round(e.start_time.to_numpy()[:20], 4)))
        if len(e) > 5 and key in fp_ev and fp_ev[key][0] != i:
            dup.append((f"events {k} / {fp_ev[key][1]}", S[fp_ev[key][0]].session, s.session))
        fp_ev[key] = (i, k)
# also: same subject, same LFP length
lens = collections.defaultdict(list)
for s in S: lens[(s.subject, s.lfp_shape[0])].append(s.session)
dup += [("same LFP length", *v) for v in lens.values() if len(v) > 1]
print("Duplicate candidates:", dup if dup else "none")
# Within one file, two event tables holding the same events (an LS table copied into another) are reported too.
same = []
for s in S:
    ks = sorted(s.events)
    for a in range(len(ks)):
        for b in range(a + 1, len(ks)):
            ea, eb = s.events[ks[a]].peak_time.to_numpy(), s.events[ks[b]].peak_time.to_numpy()
            if len(ea) and len(eb):
                j = np.clip(np.searchsorted(eb, ea), 1, len(eb) - 1)
                near = np.minimum(np.abs(eb[j] - ea), np.abs(eb[j - 1] - ea)) < 0.002
                if near.mean() > 0.9 and abs(len(ea) - len(eb)) < 0.1 * len(ea):
                    same.append((s.session[:25], ks[a], ks[b], len(ea), len(eb)))
print("Tables within a file with the same events:", same if same else "none")

In [ ]:
# One clock. (1) LS spikes around LS HFO peak times and hippocampal spikes around CA1 ripple peak times: the rate
# within +-10 ms over the rate 100-200 ms away, with the true times and with the times shifted by 1 s.
# (2) Trials against sleep scoring: the fraction of trial time scored WAKE, with the true states and with
# the states shifted by +-1800 s. A shared clock gives a ratio far above 1 at the true times only, and trials in WAKE.
def peri_ratio(s, units, times):
    c = sum(T.triggered_counts(s.spikes[u], times, 0.2, 0.01).sum(0) for u in units)
    lag = np.arange(-20, 21)
    return c[np.abs(lag) <= 1].mean() / max(c[(np.abs(lag) >= 10)].mean(), 1e-9)
clk = []
for s in S:
    ls = s.units_in(["ls"]); hp = s.units_in(HPC)
    e1, e2 = ls_ev(s), ca1_ev(s)
    r = dict(session=s.session[:25], subject=s.subject)
    if e1 is not None and len(ls):
        t = e1.peak_time.to_numpy(); r["ls_hfo"] = peri_ratio(s, ls, t); r["ls_hfo_shift1s"] = peri_ratio(s, ls, t + 1.0)
    if e2 is not None and len(hp):
        t = e2.peak_time.to_numpy(); r["ca1_rip"] = peri_ratio(s, hp, t); r["ca1_rip_shift1s"] = peri_ratio(s, hp, t + 1.0)
    if s.has_trials and s.sleep is not None:
        wk = s.state_intervals("WAKE"); tiv = s.trials[["start_time", "stop_time"]].to_numpy()
        r["trial_in_wake"] = T.total(T.intersect(tiv, wk)) / T.total(tiv)
        r["trial_in_wake_shift"] = np.mean([T.total(T.intersect(tiv, wk + d)) / T.total(tiv) for d in (-1800, 1800)])
    clk.append(r)
CLK = pd.DataFrame(clk)
print(CLK.describe().loc[["count", "min", "50%", "max"]].round(3).to_string())
print("Files where the LS HFO ratio is below 1.5:", CLK[CLK.ls_hfo < 1.5].session.tolist())
print("Files where the CA1 ripple ratio is below 1.5:", CLK[CLK.ca1_rip < 1.5].session.tolist())
print("Files where less than half the trial time is scored WAKE:", CLK[CLK.trial_in_wake < 0.5][["session", "trial_in_wake"]].values.tolist())

## Figure 7E: LS spike counts predicted from SPW-R features
Common to fig7e-ca1-spikes, fig7e-state and fig7e-duration. Recordings with LS units, hippocampal units and CA1Ripples. For each LS
neuron and feature, the error (MSE) of predicting the neuron's spike count in each SPW-R from the feature, against the MSEs of 100
shuffles of the feature across SPW-Rs (the claim's seed); the neuron counts when a two-sided one-sample t-test of the shuffled MSEs
against the actual MSE gives p < 0.05 with the actual lower. Chance: one shuffle tested against the other 99 the same way.
Variant (z reading, verifier round 1): the Figure 7E y-axis is "the number of standard deviations that the mean squared error (MSE)
was from a distribution of MSE values calculated from shuffled data", so the neuron also counts when z = (actual - shuffle mean) /
shuffle SD is below -1.96 (two-sided p < 0.05, actual lower); chance is one shuffle's z against the other 99. The claims file's
t-test of the shuffled MSEs against the actual MSE uses the standard error of the shuffle mean, so a small difference passes; the
z reading is shown beside it and asked of a reader (questions/tingley-2019.yaml).
Readings (claims file): model (a) linear regression, (b) Poisson GLM; MSE in-sample or 5-fold cross-validated (contiguous folds,
a standard choice); LS window SPW-R start to stop or +-50 ms of the ripple time; no minimum firing, or 20 spikes in SPW-Rs.
Features: hippocampal spike count (all hippocampal units, start to stop); state (NREM 1, WAKE 0, SPW-Rs in those states);
duration (stop minus start).

In [ ]:
NSHUF = 100
def fit_mse(Y, X, model, cv):
    """Y [neurons, n], X [perms, n] (feature standardized). Returns MSE [neurons, perms]."""
    n = Y.shape[1]
    folds = [np.arange(n)] if not cv else np.array_split(np.arange(n), 5)
    sse = np.zeros((Y.shape[0], X.shape[0]))
    for f in folds:
        tr = np.setdiff1d(np.arange(n), f) if cv else f
        te = f
        xt, yt = X[:, tr], Y[:, tr]
        if model == "linear":
            mx, my = xt.mean(1), yt.mean(1)
            sxy = yt @ xt.T / len(tr) - my[:, None] * mx[None, :]
            sxx = (xt ** 2).mean(1) - mx ** 2
            b = sxy / np.where(sxx > 0, sxx, np.inf)[None, :]; a = my[:, None] - b * mx[None, :]
        else:
            a = np.log(np.maximum(yt.mean(1), 1e-3))[:, None] * np.ones((1, X.shape[0])); b = np.zeros_like(a)
            for _ in range(8):
                eta = np.clip(a[:, :, None] + b[:, :, None] * xt[None], -20, 20); mu = np.exp(eta)
                r = yt[:, None, :] - mu
                ga, gb = r.sum(-1), (r * xt[None]).sum(-1)
                h11, h12, h22 = mu.sum(-1), (mu * xt[None]).sum(-1), (mu * xt[None] ** 2).sum(-1)
                det = h11 * h22 - h12 ** 2 + 1e-9
                a = a + (h22 * ga - h12 * gb) / det; b = b + (h11 * gb - h12 * ga) / det
        xs = X[:, te]
        pred = a[:, :, None] + b[:, :, None] * xs[None] if model == "linear" else np.exp(np.clip(a[:, :, None] + b[:, :, None] * xs[None], -20, 20))
        sse += ((Y[:, None, te] - pred) ** 2).sum(-1)
    return sse / n
def sig_from_mse(M):
    """M [neurons, 1 + NSHUF]; actual in column 0. Significant (actual lower, p < 0.05) and chance (shuffle 1 against the rest)."""
    act, sh = M[:, 0], M[:, 1:]
    t = stats.ttest_1samp(sh, act[:, None], axis=1)
    sig = (t.pvalue < 0.05) & (act < sh.mean(1))
    t2 = stats.ttest_1samp(sh[:, 1:], sh[:, :1], axis=1)
    ch = (t2.pvalue < 0.05) & (sh[:, 0] < sh[:, 1:].mean(1))
    # z reading: the actual MSE in SD of the shuffled MSEs (Figure 7E y-axis), two-sided p < 0.05 with the actual lower
    z = (act - sh.mean(1)) / sh.std(1, ddof=1)
    z2 = (sh[:, 0] - sh[:, 1:].mean(1)) / sh[:, 1:].std(1, ddof=1)
    return sig, ch, z < -1.96, z2 < -1.96
F7_SESS = [s for s in S if ca1_ev(s) is not None and len(s.units_in(["ls"])) and len(s.units_in(HPC))]
print(f"Recordings: {len(F7_SESS)} ({collections.Counter(s.subject for s in F7_SESS)}); LS neurons {sum(len(s.units_in(['ls'])) for s in F7_SESS)}; "
      f"CA3 by unit label {sum(len(s.units_in(['ca3'])) > 0 for s in F7_SESS)}")
def fig7_recording(s, rng, models=("linear", "poisson")):
    c = ca1_ev(s); st, sp_ = c.start_time.to_numpy(), c.stop_time.to_numpy(); tt = c.peak_time.to_numpy()
    hp = s.units_in(HPC); ls = s.units_in(["ls"])
    state = T.state_at(tt, s)
    feats = {"ca1-spikes": (sum(T.count_in(s.spikes[u], st, sp_) for u in hp).astype(float), np.ones(len(tt), bool)),
             "state": ((state == "NREM").astype(float), np.isin(state, ["NREM", "WAKE"])),
             "duration": (sp_ - st, np.ones(len(tt), bool))}
    rows = []
    for win in ["start-stop", "+-50 ms"]:
        Y = np.vstack([T.count_in(s.spikes[u], st, sp_) if win == "start-stop" else T.count_in(s.spikes[u], tt - 0.05, tt + 0.05) for u in ls]).astype(float)
        for fn, (x, m) in feats.items():
            if m.sum() < 10 or np.std(x[m]) == 0: continue
            xm = (x[m] - x[m].mean()) / x[m].std()
            X = np.vstack([xm] + [rng.permutation(xm) for _ in range(NSHUF)])
            for model in models:
                for cv in [False, True]:
                    sig, ch, sz, cz = sig_from_mse(fit_mse(Y[:, m], X, model, cv))
                    for k, u in enumerate(ls):
                        rows.append(dict(session=s.session, subject=s.subject, unit=u, feature=fn, window=win, model=model, cv=cv,
                                         n_spk=Y[k, m].sum(), sig=sig[k], chance=ch[k], sig_z=sz[k], chance_z=cz[k]))
    return rows
FAILED7 = []
def fig7(seed, models=("linear", "poisson")):
    rng = np.random.default_rng(seed)
    rows = []
    for s in F7_SESS:
        try:
            rows += fig7_recording(s, rng, models)
        except Exception as e:
            FAILED7.append((s.session, repr(e)))
    return pd.DataFrame(rows)
t0 = time.time()
F7 = fig7(seed_of("fig7e-ca1-spikes"))
print(f"computed in {time.time()-t0:.0f} s; recordings that failed: {FAILED7 if FAILED7 else 'none'}")
def f7_table(F7):
    rows = []
    for (fn, win, model, cv), g in F7.groupby(["feature", "window", "model", "cv"]):
        for mn in [0, 20]:
            gg = g[g.n_spk >= mn] if mn else g
            k, n = int(gg.sig.sum()), len(gg)
            pb = stats.binomtest(k, n, max(gg.chance.mean(), 1e-6), alternative="greater").pvalue
            kz = int(gg.sig_z.sum()); pz = stats.binomtest(kz, n, max(gg.chance_z.mean(), 1e-6), alternative="greater").pvalue
            rows.append(dict(feature=fn, window=win, model=model, cv=cv, min_spikes=mn, neurons=n, percent=100 * k / n, chance_percent=100 * gg.chance.mean(), binom_p=pb,
                             z_percent=100 * kz / n, z_chance_percent=100 * gg.chance_z.mean(), z_binom_p=pz))
    return pd.DataFrame(rows)
F7T = f7_table(F7)
EXPECT7 = {"ca1-spikes": 38, "state": 26, "duration": 20}
F7T["within_tol"] = [abs(r.percent - EXPECT7[r.feature]) <= 0.1 * EXPECT7[r.feature] for r in F7T.itertuples()]
print(F7T.to_string(float_format=lambda v: f"{v:.3g}"))
for fn, e in EXPECT7.items():
    g = F7T[F7T.feature == fn]
    print(f"{fn}: paper {e}%; combinations within tolerance {int(g.within_tol.sum())} of {len(g)}; range {g.percent.min():.1f} to {g.percent.max():.1f}%; "
          f"above chance (binomial p < 0.05) in {int((g.binom_p < 0.05).sum())}")
    print(f"   z reading: range {g.z_percent.min():.1f} to {g.z_percent.max():.1f}%; within tolerance {int((abs(g.z_percent - e) <= 0.1 * e).sum())} of {len(g)}; "
          f"chance {g.z_chance_percent.min():.1f} to {g.z_chance_percent.max():.1f}%; above chance (binomial p < 0.05) in {int((g.z_binom_p < 0.05).sum())}")
# The two tests' chance rates on synthetic data: 100 normal draws as shuffles and one more as the "actual" value, 20000 times.
_r = np.random.default_rng(seed_of("fig7e-ca1-spikes") + 1); _M = _r.normal(size=(20000, 1 + NSHUF))
_s = sig_from_mse(_M)
print(f"Synthetic null (actual drawn from the shuffle distribution): t reading {100 * _s[0].mean():.1f}%, z reading {100 * _s[2].mean():.1f}% of neurons significant")
print("\nPer animal, linear in-sample start-stop, no minimum:")
print(F7[(F7.model == "linear") & (~F7.cv) & (F7.window == "start-stop")].groupby(["feature", "subject"]).sig.agg(["mean", "count"]).round(3).to_string())

In [ ]:
# The shuffles are random: the linear-model readings under four more seeds (the Poisson fits are not repeated, for time).
for k in range(1, 5):
    sd = seed_of("fig7e-ca1-spikes") + 1000 * k
    D = f7_table(fig7(sd, models=("linear",)))
    D = D[D.min_spikes == 0]
    print(f"seed {sd}:", "; ".join(f"{fn} {g.percent.min():.1f}-{g.percent.max():.1f}% (within tolerance {int((abs(g.percent - EXPECT7[fn]) <= 0.1 * EXPECT7[fn]).sum())}; "
                                 f"z reading {g.z_percent.min():.1f}-{g.z_percent.max():.1f}%, within tolerance {int((abs(g.z_percent - EXPECT7[fn]) <= 0.1 * EXPECT7[fn]).sum())})"
                                 for fn, g in D.groupby("feature")))

**Result: fig7e-ca1-spikes partially reproduced.** Over 1288 LS neurons in the tests (of 1353 in 42 recordings), the hippocampal spike count in a SPW-R significantly predicts the LS spike count of 47.0% to 80.0% of neurons over the 16 combinations under the claims file's t-test, against 38% +- 3.8%; no combination is within tolerance. Every combination exceeds its chance rate (25.4% to 44.3%; binomial p <= 2.4e-7). The four other seeds (linear models) give 46.6% to 64.2%. That test's chance rate is high: on synthetic null data it calls 42.2% of neurons significant, against 2.5% for the z reading. Under the z reading of the Figure 7E axis (shown, and asked of a reader), 15.2% to 64.6%, within tolerance in 3 of 16 combinations, with chance 2.3% to 7.4%.

**Result: fig7e-state partially reproduced.** Under the claims file's t-test, the state (NREM or WAKE) predicts 44.4% to 63.9% of 1238 LS neurons, against 26% +- 2.6%; none within tolerance; above chance in all 16 combinations (binomial p at most 1.23e-8). Under the z reading, 16.8% to 39.5%, within tolerance in 4 of 16, chance 2.1% to 5.8%.

**Result: fig7e-duration partially reproduced.** Under the claims file's t-test, the SPW-R duration predicts 41.1% to 89.0% of 1288 LS neurons, against 20% +- 2%; none within tolerance; above chance in all 16 combinations (binomial p <= 6.3e-4). Under the z reading, 11.1% to 79.2%, within tolerance in 2 of 16, chance 1.9% to 6.2%.

## Summary

The outcomes of all claims are in the summary of `reproduce_tingley_2019.ipynb`.

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and the differences from the paper's method.